In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/anishapanja/tcga-brca-a2-clini-xlsx/TCGA-BRCA-A2-CLINI.xlsx


In [2]:
import os
print(os.getcwd())
print("Storage available:")
os.system("df -h /kaggle/working")

/kaggle/working
Storage available:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  176K   20G   1% /kaggle/working


0

In [3]:
import torch

# torch is PyTorch — the deep learning library we use throughout
# torch.cuda checks if a GPU is available and accessible

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    # If GPU exists, print its name and memory size
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", 
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 
          "GB")
else:
    print("No GPU active — running on CPU only")
    print("This is fine for data setup. Activate GPU only before training.")

GPU Available: False
No GPU active — running on CPU only
This is fine for data setup. Activate GPU only before training.


In [4]:
import os
import zipfile
import urllib.request
os.makedirs("/kaggle/working/tiles/A2", exist_ok=True)
print("Folder created successfully")
print("Ready to download A2 tiles")

Folder created successfully
Ready to download A2 tiles


In [5]:
import urllib.request

# The exact URL for the 464MB pilot file, confirmed by you from Zenodo
url = "https://zenodo.org/records/5337009/files/TCGA-BRCA-A2-DEEPMED-TILES_100.zip?download=1"

# This is where the downloaded file will be saved on Kaggle's disk
save_path = "/kaggle/working/tiles/A2/A2_pilot_100.zip"

# urllib.request.urlretrieve does the actual downloading.
# It takes the url, connects to it, and streams the file 
# into save_path. This can take a few minutes for 464MB.
print("Starting download... this may take 2-5 minutes")
urllib.request.urlretrieve(url, save_path)
print("Download complete")

# Check the file actually arrived and how big it is
import os
size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"File size on disk: {size_mb:.1f} MB")

Starting download... this may take 2-5 minutes
Download complete
File size on disk: 442.5 MB


In [6]:
import zipfile

# Path to the file we just downloaded
zip_path = "/kaggle/working/tiles/A2/A2_pilot_100.zip"

# zipfile.ZipFile opens the zip archive for reading ('r')
# We use 'with' so Python automatically closes the file 
# properly afterward, even if something goes wrong inside
with zipfile.ZipFile(zip_path, 'r') as zf:
    
    # testzip() checks every file inside the archive for 
    # corruption. It returns None if everything is fine, 
    # or the name of the first bad file if something is broken
    bad_file = zf.testzip()
    
    if bad_file is None:
        print("Zip file integrity check PASSED — safe to extract")
    else:
        print(f"CORRUPTION FOUND in: {bad_file}")
        print("We will need to re-download")
    
    # Let's also peek at how many files are inside and 
    # show the first 10 names, so we understand the 
    # folder structure before extracting everything
    file_list = zf.namelist()
    print(f"\nTotal files inside zip: {len(file_list)}")
    print("\nFirst 10 file paths inside the zip:")
    for name in file_list[:10]:
        print(" ", name)

Zip file integrity check PASSED — safe to extract

Total files inside zip: 10101

First 10 file paths inside the zip:
  BLOCKS_NORM_MACENKO/
  BLOCKS_NORM_MACENKO/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-4631-8292-A9DBCA0BD37C/
  BLOCKS_NORM_MACENKO/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-4631-8292-A9DBCA0BD37C/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-4631-8292-A9DBCA0BD37C_(100686,37368).jpg
  BLOCKS_NORM_MACENKO/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-4631-8292-A9DBCA0BD37C/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-4631-8292-A9DBCA0BD37C_(103800,39444).jpg
  BLOCKS_NORM_MACENKO/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-4631-8292-A9DBCA0BD37C/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-4631-8292-A9DBCA0BD37C_(106914,40482).jpg
  BLOCKS_NORM_MACENKO/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-4631-8292-A9DBCA0BD37C/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-4631-8292-A9DBCA0BD37C_(108990,42558).jpg
  BLOCKS_NORM_MACENKO/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-4631-8292-A9DBCA0BD37C/TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-1CF1-46

In [7]:
import zipfile

zip_path = "/kaggle/working/tiles/A2/A2_pilot_100.zip"
extract_path = "/kaggle/working/tiles/A2/extracted"

# Extract everything from the zip into extract_path
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_path)

print("Extraction complete")

# Delete the zip now — we don't need it anymore, 
# and it's wasting half our 20GB disk space
import os
os.remove(zip_path)
print("Zip deleted, space freed")

os.system("df -h /kaggle/working")

Extraction complete
Zip deleted, space freed
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  475M   20G   3% /kaggle/working


0

In [8]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [9]:
import os

# Look inside the 'datasets' folder to find your actual uploaded file
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/anishapanja/tcga-brca-a2-clini-xlsx/TCGA-BRCA-A2-CLINI.xlsx


In [10]:
import pandas as pd

# Full path to the CLINI file we just located
clini_path = "/kaggle/input/datasets/anishapanja/tcga-brca-a2-clini-xlsx/TCGA-BRCA-A2-CLINI.xlsx"

# pandas.read_excel loads an Excel file into a DataFrame — 
# think of a DataFrame as a table, like a spreadsheet, 
# that Python can manipulate row by row, column by column
clini_a2 = pd.read_excel(clini_path)

# .shape tells us (number of rows, number of columns) — 
# a quick sanity check that the file loaded correctly
print("Shape of CLINI table:", clini_a2.shape)

# We only care about a few columns right now: the patient 
# ID, the molecular subtype label, and the TIL fraction 
# we'll need later for biological validation
relevant_cols = clini_a2[["PATIENT", "TCGA Subtype", "TIL Regional Fraction"]]

# .head(10) shows the first 10 rows so we can visually 
# confirm the data looks correct — right patient ID format, 
# right subtype labels
print(relevant_cols.head(10))

Shape of CLINI table: (102, 203)
        PATIENT TCGA Subtype  TIL Regional Fraction
0  TCGA-A2-A04N    BRCA.LumA               0.264056
1  TCGA-A2-A04P   BRCA.Basal               4.796298
2  TCGA-A2-A04Q   BRCA.Basal               3.415181
3  TCGA-A2-A04R    BRCA.LumB               1.557835
4  TCGA-A2-A04T   BRCA.Basal              12.279608
5  TCGA-A2-A04U   BRCA.Basal               0.256492
6  TCGA-A2-A04V    BRCA.LumA               0.522302
7  TCGA-A2-A04W    BRCA.Her2              10.662585
8  TCGA-A2-A04X    BRCA.Her2               4.141848
9  TCGA-A2-A04Y    BRCA.LumB               6.151252


In [11]:
# The subtype column has a "BRCA." prefix on every value 
# (e.g. "BRCA.LumA") which is just noise for us — we only 
# need "LumA". str.replace removes that prefix.
clini_a2["subtype_clean"] = clini_a2["TCGA Subtype"].str.replace("BRCA.", "", regex=False)

# Now we drop any patient labeled "Normal" — these are 
# normal-adjacent tissue samples, not tumor subtypes, 
# and we excluded them from our research design earlier
clini_a2_filtered = clini_a2[clini_a2["subtype_clean"] != "Normal"].copy()

# Keep only the three columns we actually need going forward
clini_a2_filtered = clini_a2_filtered[["PATIENT", "subtype_clean", "TIL Regional Fraction"]]

# Reset the row index after filtering — purely cosmetic, 
# keeps row numbers clean and sequential (0,1,2...) instead 
# of having gaps where we dropped Normal rows
clini_a2_filtered = clini_a2_filtered.reset_index(drop=True)

print("Patients remaining after dropping Normal:", len(clini_a2_filtered))
print("\nClass distribution:")
print(clini_a2_filtered["subtype_clean"].value_counts())

print("\nFirst 5 rows:")
print(clini_a2_filtered.head())

Patients remaining after dropping Normal: 95

Class distribution:
subtype_clean
LumA     44
Basal    24
LumB     19
Her2      8
Name: count, dtype: int64

First 5 rows:
        PATIENT subtype_clean  TIL Regional Fraction
0  TCGA-A2-A04N          LumA               0.264056
1  TCGA-A2-A04P         Basal               4.796298
2  TCGA-A2-A04Q         Basal               3.415181
3  TCGA-A2-A04R          LumB               1.557835
4  TCGA-A2-A04T         Basal              12.279608


In [12]:
import os

extracted_root = "/kaggle/working/tiles/A2/extracted/BLOCKS_NORM_MACENKO"

# os.listdir gives us every folder name inside BLOCKS_NORM_MACENKO.
# Each folder name is one whole-slide image's unique ID, 
# e.g. "TCGA-A2-A04N-01Z-00-DX1.9E9B7DB0-..."
slide_folders = os.listdir(extracted_root)

print("Total slide folders found:", len(slide_folders))
print("\nFirst 5 folder names:")
for f in slide_folders[:5]:
    print(" ", f)

# The patient ID is always the first 12 characters of the 
# slide folder name, e.g. "TCGA-A2-A04N" out of 
# "TCGA-A2-A04N-01Z-00-DX1...". We extract that here.
sample_patient_id = slide_folders[0][:12]
print("\nExtracted patient ID from first folder:", sample_patient_id)

Total slide folders found: 100

First 5 folder names:
  TCGA-A2-A04W-01Z-00-DX1.F7E7B945-2ADC-4741-8FCE-ACEA657DB9C7
  TCGA-A2-A3XY-01Z-00-DX1.E57FC9BF-411E-4028-AC10-8BCA5D0C8472
  TCGA-A2-A0D3-01Z-00-DX1.D18CE1C0-291D-41C1-8F71-2A0551F8C661
  TCGA-A2-A04R-01Z-00-DX1.BE3661A3-A2A0-4248-B7D5-D8986E529B6C
  TCGA-A2-A0EW-01Z-00-DX1.F24495CB-63D8-483F-9834-F761E3F16BF0

Extracted patient ID from first folder: TCGA-A2-A04W


In [13]:
import os
import pandas as pd

extracted_root = "/kaggle/working/tiles/A2/extracted/BLOCKS_NORM_MACENKO"
slide_folders = os.listdir(extracted_root)

# We will build a list of dictionaries, one per image patch.
# Each dictionary holds: the full file path to the patch, 
# and the patient ID it belongs to. We fill in the label 
# afterward by merging with our CLINI table.
patch_records = []

for slide_folder in slide_folders:
    patient_id = slide_folder[:12]  # first 12 chars = patient ID
    slide_path = os.path.join(extracted_root, slide_folder)
    
    # List every .jpg patch inside this one slide's folder
    patch_files = [f for f in os.listdir(slide_path) if f.endswith(".jpg")]
    
    for patch_file in patch_files:
        full_patch_path = os.path.join(slide_path, patch_file)
        patch_records.append({
            "patient_id": patient_id,
            "slide_folder": slide_folder,
            "patch_path": full_patch_path
        })

# Convert our list of dictionaries into a DataFrame — 
# one row per patch, across all 100 slides
patches_df = pd.DataFrame(patch_records)

print("Total individual patches found:", len(patches_df))
print("Total unique patients (from folder names):", patches_df["patient_id"].nunique())
print("\nFirst 5 rows:")
print(patches_df.head())

Total individual patches found: 10000
Total unique patients (from folder names): 100

First 5 rows:
     patient_id                                       slide_folder  \
0  TCGA-A2-A04W  TCGA-A2-A04W-01Z-00-DX1.F7E7B945-2ADC-4741-8FC...   
1  TCGA-A2-A04W  TCGA-A2-A04W-01Z-00-DX1.F7E7B945-2ADC-4741-8FC...   
2  TCGA-A2-A04W  TCGA-A2-A04W-01Z-00-DX1.F7E7B945-2ADC-4741-8FC...   
3  TCGA-A2-A04W  TCGA-A2-A04W-01Z-00-DX1.F7E7B945-2ADC-4741-8FC...   
4  TCGA-A2-A04W  TCGA-A2-A04W-01Z-00-DX1.F7E7B945-2ADC-4741-8FC...   

                                          patch_path  
0  /kaggle/working/tiles/A2/extracted/BLOCKS_NORM...  
1  /kaggle/working/tiles/A2/extracted/BLOCKS_NORM...  
2  /kaggle/working/tiles/A2/extracted/BLOCKS_NORM...  
3  /kaggle/working/tiles/A2/extracted/BLOCKS_NORM...  
4  /kaggle/working/tiles/A2/extracted/BLOCKS_NORM...  


In [14]:
# Merge patches_df with our cleaned CLINI label table.
# how="left" means: keep every row in patches_df (the patches),
# and attach matching label info from clini_a2_filtered where 
# patient_id matches PATIENT. If a patient has no match 
# (e.g. they were dropped as "Normal"), their label columns 
# will show as NaN (missing) instead of crashing.
patches_labeled = patches_df.merge(
    clini_a2_filtered,
    left_on="patient_id",
    right_on="PATIENT",
    how="left"
)

# Find patches whose patient_id did NOT find a match — 
# these are the "extra 5" patients we noticed earlier
unmatched = patches_labeled[patches_labeled["subtype_clean"].isna()]
unmatched_patients = unmatched["patient_id"].unique()

print("Patients with NO subtype label (will be dropped):")
print(unmatched_patients)
print("\nNumber of patches belonging to these unmatched patients:", len(unmatched))

# Now keep only patches that DID get a valid label
patches_final = patches_labeled[patches_labeled["subtype_clean"].notna()].copy()
patches_final = patches_final.reset_index(drop=True)

print("\n--- FINAL CLEAN DATASET ---")
print("Total labeled patches:", len(patches_final))
print("Total unique patients:", patches_final["patient_id"].nunique())
print("\nPatch count per class:")
print(patches_final["subtype_clean"].value_counts())

Patients with NO subtype label (will be dropped):
['TCGA-A2-A1G6' 'TCGA-A2-A0YK' 'TCGA-A2-A4RY' 'TCGA-A2-A0CZ'
 'TCGA-A2-A3XW' 'TCGA-A2-A25A' 'TCGA-A2-A0CL']

Number of patches belonging to these unmatched patients: 700

--- FINAL CLEAN DATASET ---
Total labeled patches: 9300
Total unique patients: 93

Patch count per class:
subtype_clean
LumA     4400
Basal    2300
LumB     1900
Her2      700
Name: count, dtype: int64


In [15]:
# Check the ORIGINAL unfiltered clini_a2 (before we dropped 
# Normal) for these 7 patient IDs, to see what subtype 
# they actually were
check = clini_a2[clini_a2["PATIENT"].isin(unmatched_patients)][["PATIENT", "TCGA Subtype"]]
print(check)

         PATIENT TCGA Subtype
11  TCGA-A2-A0CL  BRCA.Normal
24  TCGA-A2-A0CZ  BRCA.Normal
65  TCGA-A2-A0YK  BRCA.Normal
76  TCGA-A2-A1G6  BRCA.Normal
78  TCGA-A2-A25A  BRCA.Normal
90  TCGA-A2-A3XW  BRCA.Normal
97  TCGA-A2-A4RY  BRCA.Normal


In [16]:
import shutil
import os

pilot_folder = "/kaggle/working/tiles/A2/extracted"

# shutil.rmtree deletes a folder and everything inside it, 
# recursively. We're removing the pilot patches now since 
# we've already confirmed the pipeline works correctly — 
# we don't need this data anymore, and we need the disk space.
shutil.rmtree(pilot_folder)

print("Pilot folder removed")
os.system("df -h /kaggle/working")

Pilot folder removed
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  192K   20G   1% /kaggle/working


0

In [17]:
import os

# Check every file currently sitting in /kaggle/working 
# and how big each one is — this tells us exactly what 
# state we're in right now, whether downloads completed, 
# are stuck, or got duplicated
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        full_path = os.path.join(root, file)
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f"{size_mb:8.1f} MB   {full_path}")

print("\n--- Disk summary ---")
os.system("df -h /kaggle/working")

     0.0 MB   /kaggle/working/__notebook__.ipynb

--- Disk summary ---
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  192K   20G   1% /kaggle/working


0

In [18]:
import os

# Remove the truncated, broken zip — it's useless at 39MB 
# out of an expected 12.8GB, just a partial fragment
broken_path = "/kaggle/working/A2_full.zip"
if os.path.exists(broken_path):
    os.remove(broken_path)
    print("Broken partial file removed")

print("Disk usage now:")
os.system("df -h /kaggle/working")

Disk usage now:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  192K   20G   1% /kaggle/working


0

In [19]:
import urllib.request
import os

url = "https://zenodo.org/records/5337009/files/TCGA-BRCA-A2-DEEPMED-TILES.zip?download=1"
zip_path = "/kaggle/working/A2_full.zip"

# Zenodo told us this file is 12.8 GB. We use this as a 
# sanity check afterward — if what we downloaded is 
# wildly smaller than this, something went wrong and 
# we should retry rather than silently continuing 
# with broken data
expected_min_size_gb = 12.0

max_attempts = 3

for attempt in range(1, max_attempts + 1):
    print(f"--- Attempt {attempt} of {max_attempts} ---")
    try:
        print("Downloading... please keep this tab open and active")
        urllib.request.urlretrieve(url, zip_path)
        
        size_gb = os.path.getsize(zip_path) / (1024**3)
        print(f"Download finished. Size on disk: {size_gb:.2f} GB")
        
        if size_gb >= expected_min_size_gb:
            print("Size looks correct — download successful")
            break
        else:
            print("File is too small — likely interrupted. Retrying...")
            os.remove(zip_path)
            
    except Exception as e:
        print("Download failed with error:", e)
        if os.path.exists(zip_path):
            os.remove(zip_path)

else:
    print("\nAll attempts failed. Do not proceed to extraction.")

--- Attempt 1 of 3 ---
Downloading... please keep this tab open and active
Download finished. Size on disk: 11.90 GB
File is too small — likely interrupted. Retrying...
--- Attempt 2 of 3 ---
Downloading... please keep this tab open and active
Download finished. Size on disk: 11.90 GB
File is too small — likely interrupted. Retrying...
--- Attempt 3 of 3 ---
Downloading... please keep this tab open and active
Download finished. Size on disk: 11.90 GB
File is too small — likely interrupted. Retrying...

All attempts failed. Do not proceed to extraction.


In [20]:
import urllib.request
import os
import zipfile

url = "https://zenodo.org/records/5337009/files/TCGA-BRCA-A2-DEEPMED-TILES.zip?download=1"
zip_path = "/kaggle/working/A2_full.zip"

# Remove anything broken/partial sitting there from before
if os.path.exists(zip_path):
    os.remove(zip_path)

print("Downloading full A2 dataset... keep this tab active")
urllib.request.urlretrieve(url, zip_path)

size_gb = os.path.getsize(zip_path) / (1024**3)
print(f"Download finished. Size on disk: {size_gb:.2f} GB")

# Instead of guessing whether this size is "correct", we 
# directly ask the zip file whether it is structurally 
# complete and uncorrupted. This is more reliable than 
# any size estimate, because it checks the actual file 
# contents rather than a number we might get wrong.
print("Verifying zip integrity...")
with zipfile.ZipFile(zip_path, 'r') as zf:
    bad_file = zf.testzip()
    if bad_file is None:
        print("Integrity check PASSED — file is complete and safe to use")
    else:
        print(f"CORRUPTION FOUND in: {bad_file} — need to redownload")

Download finished. Size on disk: 11.90 GB
Verifying zip integrity...
Integrity check PASSED — file is complete and safe to use


In [21]:
import zipfile
import os
import random
from collections import defaultdict

zip_path = "/kaggle/working/A2_full.zip"
extract_path = "/kaggle/working/tiles/A2/extracted_full"
PATCHES_PER_PATIENT = 200

random.seed(42)

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    
    all_names = zf.namelist()
    jpg_names = [n for n in all_names if n.endswith(".jpg")]
    print("Total patches available in full A2 zip:", len(jpg_names))
    
    patches_by_patient = defaultdict(list)
    for name in jpg_names:
        parts = name.split("/")
        slide_id = parts[1]
        patient_id = slide_id[:12]
        patches_by_patient[patient_id].append(name)
    
    print("Total unique patients found:", len(patches_by_patient))
    
    extracted_count = 0
    for patient_id, patch_list in patches_by_patient.items():
        if len(patch_list) <= PATCHES_PER_PATIENT:
            selected = patch_list
        else:
            selected = random.sample(patch_list, PATCHES_PER_PATIENT)
        
        for patch_name in selected:
            zf.extract(patch_name, extract_path)
            extracted_count += 1
    
    print("Total patches actually extracted:", extracted_count)

print("\nDisk usage now:")
os.system("df -h /kaggle/working")

Total patches available in full A2 zip: 268938
Total unique patients found: 100
Total patches actually extracted: 20000

Disk usage now:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   13G  6.7G  66% /kaggle/working


0

In [22]:
import os

zip_path = "/kaggle/working/A2_full.zip"
os.remove(zip_path)
print("A2 zip deleted")

print("\nDisk usage now:")
os.system("df -h /kaggle/working")

A2 zip deleted

Disk usage now:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  966M   19G   5% /kaggle/working


0

In [23]:
import urllib.request
import os
import zipfile

url_e2 = "https://zenodo.org/records/5337009/files/TCGA-BRCA-E2-DEEPMED-TILES.zip?download=1"
zip_path_e2 = "/kaggle/working/E2_full.zip"

if os.path.exists(zip_path_e2):
    os.remove(zip_path_e2)

print("Downloading full E2 dataset... keep this tab active")
urllib.request.urlretrieve(url_e2, zip_path_e2)

size_gb = os.path.getsize(zip_path_e2) / (1024**3)
print(f"Download finished. Size on disk: {size_gb:.2f} GB")

print("Verifying zip integrity...")
with zipfile.ZipFile(zip_path_e2, 'r') as zf:
    bad_file = zf.testzip()
    if bad_file is None:
        print("Integrity check PASSED — file is complete and safe to use")
    else:
        print(f"CORRUPTION FOUND in: {bad_file} — need to redownload")

Download finished. Size on disk: 10.31 GB
Verifying zip integrity...
Integrity check PASSED — file is complete and safe to use


In [24]:
import zipfile
import os
import random
from collections import defaultdict

zip_path = "/kaggle/working/E2_full.zip"
extract_path = "/kaggle/working/tiles/E2/extracted_full"
PATCHES_PER_PATIENT = 200

random.seed(42)

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    
    all_names = zf.namelist()
    jpg_names = [n for n in all_names if n.endswith(".jpg")]
    print("Total patches available in full E2 zip:", len(jpg_names))
    
    patches_by_patient = defaultdict(list)
    for name in jpg_names:
        parts = name.split("/")
        slide_id = parts[1]
        patient_id = slide_id[:12]
        patches_by_patient[patient_id].append(name)
    
    print("Total unique patients found:", len(patches_by_patient))
    
    extracted_count = 0
    for patient_id, patch_list in patches_by_patient.items():
        if len(patch_list) <= PATCHES_PER_PATIENT:
            selected = patch_list
        else:
            selected = random.sample(patch_list, PATCHES_PER_PATIENT)
        
        for patch_name in selected:
            zf.extract(patch_name, extract_path)
            extracted_count += 1
    
    print("Total patches actually extracted:", extracted_count)

print("\nDisk usage now:")
os.system("df -h /kaggle/working")

Total patches available in full E2 zip: 214210
Total unique patients found: 90
Total patches actually extracted: 18000

Disk usage now:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   13G  7.4G  63% /kaggle/working


0

In [25]:
import os

zip_path_e2 = "/kaggle/working/E2_full.zip"
os.remove(zip_path_e2)
print("E2 zip deleted")

print("\nDisk usage now:")
os.system("df -h /kaggle/working")

E2 zip deleted

Disk usage now:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  1.9G   18G  10% /kaggle/working


0